In [ ]:
import time
import json
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image
import os
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from pc import PS
from modules import ADC,DAC,CHIP,SELECT
from command import CMD,CmdData,Packet
from command.singleCmdInfo import *

from util import plot_v_cond,plot_cond,show_crossbar,DataLoader
import pickle
from network.layer import Layer,hnnLayer

In [ ]:
chip=CHIP(PS(host="192.168.1.11", port = 7, debug=0),init=True)
chip.set_device_cfg(deviceType=0,IsNew32=True)
chip.adc.set_gap(adc_cs_gap=100,adc_first_gap=20,adc_last_gap=20)
chip.adc.set_gain_resistor(big_resistance=22e-3,small_resistance=200)
chip.clk_manager.set_cyc(delay1=20,delay2=20,delay3=50)
chip.add_compiler("../compiler/code/")
# chip.compensation.initop("../chip_data/chip_1_4/")
# chip.compensation.initop("../chip_data/chip6_/")

In [ ]:
weight = np.load("../data/single_layer_mlp/fc1.weight.npy")
bias = np.load("../data/single_layer_mlp/fc1.bias.npy")
mlp=hnnLayer(chip=chip,weight_target=weight,weight_min=-0.17,weight_max=0.17,cond_min=0,cond_max=1100,cond_reference=550)

In [11]:
with open("../data/good_device/tg_cond.pkl", "rb") as file:
    tg_cond_dict = pickle.load(file)
tg_map = np.array(list(tg_cond_dict.keys()))
cond_map = np.array(list(tg_cond_dict.values()))
slope,intercept = np.polyfit(cond_map,tg_map, 1)

### 写权重

In [ ]:
chip.ps.receive_packet(4096)
select_point = SELECT()
# selcet_point.init_point(reset_path="../chip_data/chip6_/cond_250.npy",set_path="../chip_data/chip6_/cond_500.npy",value_stack_off=0.2,value_stack_on=-0.2)
row = [i for i in range(0,256)]
col1 = [j for j in range(0, 40)]
col2 = [j for j in range(40, 80)]
col3 = [j for j in range(80, 120)]
col4 = [j for j in range(120, 160)]
b1_row, b1_col = select_point.find_good_device(row, col1, 256, 10)
sub_matrix = select_point.good_point[np.ix_(b1_row, b1_col)]
print(np.sum(sub_matrix))
plot_cond(sub_matrix,vmax=1)
select_point.save_pos(row=b1_row,col=b1_col,file_path="../data/single_layer_mlp/pos/block1")
b2_row, b2_col = select_point.find_good_device(row, col2, 256, 10)
sub_matrix = select_point.good_point[np.ix_(b2_row, b2_col)]
print(np.sum(sub_matrix))
plot_cond(sub_matrix,vmax=1)
select_point.save_pos(row=b2_row,col=b2_col,file_path="../data/single_layer_mlp/pos/block2")
b3_row, b3_col = select_point.find_good_device(row, col3, 256, 10)
sub_matrix = select_point.good_point[np.ix_(b3_row, b3_col)]
print(np.sum(sub_matrix))
plot_cond(sub_matrix,vmax=1)
select_point.save_pos(row=b3_row,col=b3_col,file_path="../data/single_layer_mlp/pos/block3")
b4_row, b4_col = select_point.find_good_device(row, col4, 16, 10)
sub_matrix = select_point.good_point[np.ix_(b4_row, b4_col)]
print(np.sum(sub_matrix))
plot_cond(sub_matrix,vmax=1)
select_point.save_pos(row=b4_row,col=b4_col,file_path="../data/single_layer_mlp/pos/block4")

In [ ]:
threshold = 30

set_v,reset_v = 1,1
reset_pulse_width,set_pulse_width = 10e-6,100e-6
need_read = np.zeros((256,256),dtype=bool)
target = np.ones((256,256))

pos = mlp.get_weight_pos()
need_read[pos] = True
target[pos] = mlp.get_target_cond()
tg_v = target*slope+intercept-0.2
for k in range(1):
    for i in range(40):
        voltage_base = chip.read_point2(crossbar=need_read,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
        voltage = chip.read_point2(crossbar=need_read,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
        # resistence = chip.voltage_to_resistance(voltage = voltage-voltage_base)
        cond = chip.voltage_to_cond(voltage = voltage-voltage_base)
        # cond = chip.compensation.compensation_point(resistence=resistence,from_row=True,return_type=0)

        condition_reset = (cond > (target+threshold)) & need_read
        condition_set = (cond < (target-threshold)) & need_read
        plot_cond(cond[pos]-hnn.cond_reference,vmin=-hnn.cond_range,vmax=hnn.cond_range,title=f"{i},reset:{np.sum(condition_reset)},set:{np.sum(condition_set)}")

        if i>0:
            # set_pulse_width = set_pulse_width+10e-6*i
            # reset_pulse_width = reset_pulse_width+10e-6*i
            # set_v+=0.1
            # # reset_v+=0.05
            # tg_v[condition_set] +=0.025
            if i<10:
                tg_v[condition_reset] -= 0.04
                tg_v[condition_set] += 0.04
            else:
                tg_v[condition_reset] -= 0.02
                tg_v[condition_set] += 0.02

        tg_v.clip(0,3,out=tg_v)

        # chip.write_point2(crossbar=condition_reset,write_voltage=reset_v,tg=5,pulse_width=reset_pulse_width,set_device=False)
        # chip.write_point2(crossbar=condition_reset,write_voltage=set_v-0.5,tg=tg_v,pulse_width=set_pulse_width,set_device=True)
        # # # chip.ps.set_time_out(100)
        chip.write_point2(crossbar=condition_set,write_voltage=set_v,tg=tg_v,pulse_width=set_pulse_width,set_device=True)

    set_v = set_v+0.1
    reset_v = reset_v+0.1
    set_pulse_width = set_pulse_width+10e-6*k
    reset_pulse_width = reset_pulse_width+10e-6*k
    tg_v = target*slope+intercept+k*0.05-0.3